t-Testing

In [148]:
import pandas as pd
import numpy as np
from scipy import stats

Load the Data

In [149]:
df = pd.read_csv('../../output/processed_data/05_filtered.csv')

Daily Abnormal Returns

In [150]:
# AR was computed and sanity-checked in NB05.
# It is loaded directly from 05_filtered.csv — no need to recompute here.
assert 'AR' in df.columns, \
    "AR column missing from 05_filtered.csv — re-run NB05 first."
print(f'✓ AR loaded from 05_filtered.csv  ({df["AR"].notna().sum()} non-null values)')

✓ AR loaded from 05_filtered.csv  (51576 non-null values)


Event Window

In [151]:
windows = {
    'CAR(-5,-1)': (-5, -1),
    'CAR(0,0)': (0, 0),
    'CAR(0,+5)': (0, 5),
    'CAR(-5,+30)': (-5, 30)
}

t-test Function

In [152]:
def run_cross_sectional_ttest(data, start_day, end_day, window_name):
    #Filter the data
    window_data = data[(data['day'] >= start_day) & (data['day'] <= end_day)]

    #sum daily ARto get CAR
    stock_cars = window_data.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
    stock_cars.rename(columns={'AR': 'CAR'}, inplace=True)

    results = []

    # Sector level t-test
    sectors = stock_cars['sector'].unique()

    for sector in sectors:
        sector_data = stock_cars[stock_cars['sector'] == sector]['CAR']
        n_stocks = len(sector_data)
        
        if n_stocks > 1:
            mean_car = sector_data.mean()
            std_car = sector_data.std()
            t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
        else:
            mean_car = sector_data.mean()
            std_car = np.nan
            t_stat = np.nan
            p_val = np.nan

        results.append({
            'Window': window_name,
            'Level': 'Sector',
            'Name': sector,
            'N': n_stocks,
            'Mean_CAR(%)': mean_car * 100, # Convert to percentage for readability
            'Std_Dev(%)': std_car * 100,
            't-statistic': t_stat,
            'p-value': p_val,
            'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
        })

    #Market t-test
    all_cars = stock_cars['CAR']
    n_total = len(all_cars)
    t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_cars, popmean=0)

    results.append({
        'Window': window_name,
        'Level': 'Market',
        'Name': 'ALL STOCKS',
        'N': n_total,
        'Mean_CAR(%)': all_cars.mean() * 100, 
        'Std_Dev(%)': all_cars.std() * 100,
        't-statistic': t_stat_mkt,
        'p-value': p_val_mkt,
        'Significant_at_5%': p_val_mkt < 0.05 
    })

    return pd.DataFrame(results)



Test all windows

In [153]:
all_results = []

for window_name, (start, end) in windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    all_results.append(window_results_df)

final_ttest_results = pd.concat(all_results,ignore_index=True)

formatted_results = final_ttest_results.copy()
formatted_results['Mean_CAR(%)'] = formatted_results['Mean_CAR(%)'].round(2)
formatted_results['Std_Dev(%)'] = formatted_results['Std_Dev(%)'].round(2)
formatted_results['t-statistic'] = formatted_results['t-statistic'].round(3)
formatted_results['p-value'] = formatted_results['p-value'].round(4)

RQ3: Anticipation effect — did the market react before landfall?

In [154]:
anticipation = final_ttest_results[
    (final_ttest_results['Window'] == 'CAR(-5,-1)') &
    (final_ttest_results['Level']  == 'Market')
].iloc[0]

print('RQ3 — Anticipation Effect: CAR(-5,-1) market-wide')
print(f'  Mean CAR   : {anticipation["Mean_CAR(%)"]:.3f}%')
print(f'  t-statistic: {anticipation["t-statistic"]:.3f}')
print(f'  p-value    : {anticipation["p-value"]:.4f}')
print()
if anticipation['p-value'] < 0.05:
    print('  ✓ Statistically significant anticipation effect detected.')
    print('    The market was already pricing in disaster risk before landfall.')
else:
    print('  ✗ No statistically significant anticipation effect detected.')
print()

# Which sectors anticipated earliest?
print('Sector-level anticipation — CAR(-5,-1) sorted by mean CAR:')
sector_anticipation = (
    final_ttest_results[
        (final_ttest_results['Window'] == 'CAR(-5,-1)') &
        (final_ttest_results['Level']  == 'Sector')
    ]
    [['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']]
    .sort_values('Mean_CAR(%)')
)
print(sector_anticipation.to_string(index=False))

RQ3 — Anticipation Effect: CAR(-5,-1) market-wide
  Mean CAR   : -0.560%
  t-statistic: -2.128
  p-value    : 0.0346

  ✓ Statistically significant anticipation effect detected.
    The market was already pricing in disaster risk before landfall.

Sector-level anticipation — CAR(-5,-1) sorted by mean CAR:
                      Name  N  Mean_CAR(%)  t-statistic  p-value  Significant_at_5%
                    Energy  3    -6.189744    -1.882075 0.200542              False
            Transportation  2    -5.570161    -8.965764 0.070713              False
Telecommunication Services  2    -3.033591  -170.101306 0.003743               True
       Commercial Services  3    -2.887530    -0.877121 0.472926              False
                 Retailing  8    -2.841804    -1.942698 0.093164              False
               Real Estate 11    -2.376540    -2.148853 0.057175              False
                 Utilities  9    -1.825882    -1.730685 0.121755              False
       Software & Ser

In [155]:
# ── Recovery evidence — CAR(-5,+30) significance per sector ──────────────────
# Pairs with the recovery_time.csv from NB05.
# Sectors with p < 0.05 have statistically confirmed persistent damage.
# Sectors with p > 0.05 did not sustain significant full-window losses —
# consistent with rapid recovery.

assert 'final_ttest_results' in dir(), \
    "Run the 'Test all windows' cell above first before running this cell."

print('Recovery evidence — CAR(-5,+30) significance by sector:')
print('Significant (p<0.05) = statistically confirmed persistent damage\n')

rec_ev = (
    final_ttest_results[
        (final_ttest_results['Window'] == 'CAR(-5,+30)') &
        (final_ttest_results['Level']  == 'Sector')
    ]
    [['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']]
    .sort_values('Mean_CAR(%)')
    .reset_index(drop=True)
)
print(rec_ev.to_string(index=False))

rec_ev.to_csv('../../output/tables/recovery_evidence.csv', index=False)
print('\n✓ Saved recovery_evidence.csv')

Recovery evidence — CAR(-5,+30) significance by sector:
Significant (p<0.05) = statistically confirmed persistent damage

                      Name  N  Mean_CAR(%)  t-statistic  p-value  Significant_at_5%
                    Energy  3   -17.926195    -1.187104 0.357073              False
                 Insurance  9   -12.073077    -4.523183 0.001942               True
                 Utilities  9   -11.665490    -2.699728 0.027085               True
            Transportation  2   -10.088982   -13.448752 0.047250               True
         Consumer Services 20    -9.285261    -4.914378 0.000096               True
               Real Estate 11    -8.321139    -1.555915 0.150783              False
    Diversified Financials 29    -6.322856    -2.901309 0.007160               True
        Household Products  1    -4.355073          NaN      NaN              False
                     Banks 16    -2.147783    -1.129950 0.276243              False
      Healthcare Equipment  7    -0.71

In [156]:
# =============================================================================
# Extended windows — granular breakdown for paper Section 3.X
# =============================================================================
extended_windows = {
    'Pre-Event [-5,-1]' : (-5,  -1),
    'Early anticipation [-5,-3]' : (-5,  -3),
    'Late anticipation  [-2,-1]' : (-2,  -1),
    'Landfall day       [0,0]'   : ( 0,   0),
    'Acute shock        [0,+2]'  : ( 0,  +2),
    'Immediate aftermath[0,+5]'  : ( 0,  +5),
    'Short recovery     [+6,+10]': ( +6, +10),
    'Mid recovery      [+11,+20]': (+11, +20),
    'Late recovery     [+21,+30]': (+21, +30),
    'Full recovery     [-5,+30]'  : (-5,  +30)
}

extended_results = []
for window_name, (start, end) in extended_windows.items():
    window_results_df = run_cross_sectional_ttest(df, start, end, window_name)
    extended_results.append(window_results_df)

extended_ttest_df = pd.concat(extended_results, ignore_index=True)

# ── Market-level summary across all extended windows ─────────────────────────
market_extended = (
    extended_ttest_df[extended_ttest_df['Level'] == 'Market']
    [['Window','N','Mean_CAR(%)','Std_Dev(%)','t-statistic','p-value','Significant_at_5%']]
    .reset_index(drop=True)
)
market_extended['Mean_CAR(%)']  = market_extended['Mean_CAR(%)'].round(4)
market_extended['t-statistic']  = market_extended['t-statistic'].round(4)
market_extended['p-value']      = market_extended['p-value'].round(4)

print("Market-level CAR across extended windows:")
print(market_extended.to_string(index=False))
market_extended.to_csv('../../output/tables/extended_market_car.csv', index=False)

# ── Sector-level pivot: mean CAR(%) per sector per window ────────────────────
sector_extended = extended_ttest_df[extended_ttest_df['Level'] == 'Sector'].copy()
sector_extended['Mean_CAR(%)'] = sector_extended['Mean_CAR(%)'].round(4)

pivot = sector_extended.pivot(
    index='Name', columns='Window', values='Mean_CAR(%)')

# Order columns chronologically
col_order = list(extended_windows.keys())
pivot = pivot[[c for c in col_order if c in pivot.columns]]

print("\nSector CAR pivot across extended windows:")
print(pivot.to_string())
pivot.to_csv('../../output/tables/extended_sector_car_pivot.csv')

# ── Recovery speed — trough day per sector ───────────────────────────────────
print("\nRecovery Speed Analysis:")

recovery_rows = []
for sector in df['sector'].unique():
    s_df = df[
        (df['sector'] == sector) &
        (df['day'] >= 0) &
        (df['day'] <= 30)
    ].copy()

    if s_df.empty:
        continue

    daily_cum = (
        s_df.groupby('day')['AR']
            .mean()
            .cumsum()
            .reset_index()
    )
    daily_cum.columns = ['day', 'cum_AR']
    daily_cum['cum_AR_pct'] = daily_cum['cum_AR'] * 100

    trough_idx  = daily_cum['cum_AR_pct'].idxmin()
    trough_day  = daily_cum.loc[trough_idx, 'day']
    trough_val  = daily_cum.loc[trough_idx, 'cum_AR_pct']

    end_row     = daily_cum[daily_cum['day'] == 30]
    end_val     = float(end_row['cum_AR_pct'].values[0]) if len(end_row) > 0 else np.nan
    recovered   = end_val >= 0 if not np.isnan(end_val) else False

    recovery_rows.append({
        'Sector'              : sector,
        'Trough Day'          : int(trough_day),
        'Trough CAR (%)'      : round(trough_val, 4),
        'End CAR Day+30 (%)' : round(end_val, 4),
        'Recovered by Day+30' : recovered,
    })

recovery_df = (
    pd.DataFrame(recovery_rows)
      .sort_values('Trough Day')
      .reset_index(drop=True)
)
print(recovery_df.to_string(index=False))
recovery_df.to_csv('../../output/tables/sector_recovery_speed.csv', index=False)
print('\n✓ Saved extended_market_car.csv, extended_sector_car_pivot.csv, sector_recovery_speed.csv')

Market-level CAR across extended windows:
                     Window   N  Mean_CAR(%)  Std_Dev(%)  t-statistic  p-value  Significant_at_5%
          Pre-Event [-5,-1] 203      -0.5605    3.753455      -2.1276   0.0346               True
 Early anticipation [-5,-3] 203      -0.3852    3.382029      -1.6226   0.1062              False
 Late anticipation  [-2,-1] 203      -0.1753    2.110726      -1.1835   0.2380              False
   Landfall day       [0,0] 202      -0.1251    2.327170      -0.7642   0.4457              False
  Acute shock        [0,+2] 203      -1.2581    4.487918      -3.9940   0.0001               True
  Immediate aftermath[0,+5] 203      -2.0439    6.022843      -4.8352   0.0000               True
Short recovery     [+6,+10] 202       0.5265    4.455433       1.6794   0.0946              False
Mid recovery      [+11,+20] 203      -0.7496    5.520970      -1.9346   0.0544              False
Late recovery     [+21,+30] 203       0.2566    9.547204       0.3829   0.70

Results

In [157]:
display_df = formatted_results[formatted_results['Window'] == 'CAR(-5,+30)'].sort_values(by='Mean_CAR(%)')
print(display_df[['Name', 'N', 'Mean_CAR(%)', 't-statistic', 'p-value', 'Significant_at_5%']])


formatted_results.to_csv('../../output/tables/ttest_results.csv', index=False)

                          Name    N  Mean_CAR(%)  t-statistic  p-value  \
77                      Energy    3       -17.93       -1.187   0.3571   
60                   Insurance    9       -12.07       -4.523   0.0019   
76                   Utilities    9       -11.67       -2.700   0.0271   
72              Transportation    2       -10.09      -13.449   0.0472   
66           Consumer Services   20        -9.29       -4.914   0.0001   
68                 Real Estate   11        -8.32       -1.556   0.1508   
62      Diversified Financials   29        -6.32       -2.901   0.0072   
69          Household Products    1        -4.36          NaN      NaN   
79                  ALL STOCKS  203        -2.57       -2.519   0.0125   
61                       Banks   16        -2.15       -1.130   0.2762   
67        Healthcare Equipment    7        -0.72       -0.169   0.8712   
78                 Automobiles    1        -0.11          NaN      NaN   
63               Capital Goods   27   